# 02 — Matching: Empresa → Aromas
```bash
pip install groq sentence-transformers qdrant-client pandas numpy
```

## 0. Configuración

In [5]:
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
from collections import defaultdict
from groq import Groq
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

ENV_PATH = Path(".env")
if ENV_PATH.exists():
    for line in ENV_PATH.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip().strip("'\""))
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
GROQ_MODEL   = "llama-3.3-70b-versatile"

N_ATRIBUTOS        = 5    
TOP_K_DESCRIPTORES = 4    
TOP_N_MOLECULAS    = 8    

COLLECTION = "dravnieks_descriptors"

print("Configuración lista.")

Configuración lista.


## 1. Cargar recursos

In [6]:
descriptor_embeddings = np.load("descriptor_embeddings.npy")
descriptors = pd.read_csv("descriptor_list.csv")["descriptor"].tolist()

behavior_named = pd.read_csv("dravnieks_behavior_named.csv", index_col=0)

print(f"Descriptores: {len(descriptors)}")
print(f"Embeddings shape: {descriptor_embeddings.shape}")
print(f"Moléculas en dataset: {len(behavior_named)}")

Descriptores: 146
Embeddings shape: (146, 384)
Moléculas en dataset: 160


In [7]:
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
print(f"Cargando {MODEL_NAME}...")
embed_model = SentenceTransformer(MODEL_NAME)
print("Listo.")

Cargando paraphrase-multilingual-MiniLM-L12-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Listo.


In [8]:
qdrant = QdrantClient(":memory:")
VECTOR_SIZE = descriptor_embeddings.shape[1]

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE)
)
qdrant.upsert(
    collection_name=COLLECTION,
    points=[
        PointStruct(id=i, vector=descriptor_embeddings[i].tolist(),
                    payload={"descriptor": descriptors[i]})
        for i in range(len(descriptors))
    ]
)
print(f"Qdrant listo: {len(descriptors)} descriptores indexados.")

Qdrant listo: 146 descriptores indexados.


## 2. Paso 1 — extraemos atributos con pesos

In [9]:
def extraer_atributos_con_pesos(texto: str, n: int = N_ATRIBUTOS) -> list[dict]:
    """
    Dado texto de empresa, devuelve lista de dicts:
    [{'atributo': 'warm wood', 'peso': 0.35, 'razon': '...'}, ...]
    Los pesos suman 1.0.
    """
    client = Groq(api_key=GROQ_API_KEY, timeout=30.0)

    system_prompt = """You are an expert in sensory branding and olfactory marketing.
Analyze company identity texts and extract the core sensory/emotional attributes
that could be translated into olfactory experiences.

Rules:
- Output ONLY valid JSON, no other text, no markdown fences
- Each attribute must be in ENGLISH, 2-4 words, sensory and specific
- Attributes must be derived exclusively from the company text — never invent generic ones
- Good attributes: evoke a concrete smell, texture, temperature or natural element grounded in the brand's identity, ingredients, environment or values
- Bad attributes: abstract corporate concepts, generic buzzwords or anything not directly inferable from the text
- Weights must sum to exactly 1.0
- 'razon' explains briefly which part of the text inspired this attribute (in Spanish is fine)"""

    user_prompt = f"""Extract exactly {n} olfactory attributes with importance weights from this company text.

Company text:
{texto}

Return ONLY a JSON array (use your own attributes derived from the text above, not placeholders):
[
  {{"atributo": "...", "peso": 0.XX, "razon": "..."}},
  ...
]"""

    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        temperature=0.2,
        max_tokens=600
    )

    raw = response.choices[0].message.content.strip()

    raw = raw.replace("```json", "").replace("```", "").strip()

    try:
        atributos = json.loads(raw)
    except json.JSONDecodeError:
        import re
        match = re.search(r'\[.*?\]', raw, re.DOTALL)
        if match:
            atributos = json.loads(match.group())
        else:
            raise ValueError(f"No se pudo parsear JSON:\n{raw}")

    total = sum(a['peso'] for a in atributos)
    for a in atributos:
        a['peso'] = round(a['peso'] / total, 4)

    return atributos

In [10]:
# ── Texto de empresa ─────────────────────────────────────────────────────────

texto_empresa = """
Somos una marca de cosmética natural fundada en Galicia.
Nuestra misión es reconectar a las personas con la naturaleza a través de productos
elaborados con ingredientes atlánticos puros: algas marinas, líquenes de montaña,
aceites de camelia. Creemos en la calma, la autenticidad y el respeto profundo
por los ecosistemas. Nuestros valores son la transparencia, la sostenibilidad
y el cuidado lento. Cada producto es un ritual de conexión con lo salvaje y lo esencial.
"""

print("Extrayendo atributos con Groq...")
atributos = extraer_atributos_con_pesos(texto_empresa, n=N_ATRIBUTOS)

print(f"\n{len(atributos)} atributos extraídos:\n")
for a in atributos:
    print(f"  [{a['peso']:.2f}]  {a['atributo']:25s}  → {a['razon']}")

Extrayendo atributos con Groq...

5 atributos extraídos:

  [0.25]  Marine Algae               → ingredientes atlánticos puros: algas marinas
  [0.20]  Mountain Moss              → líquenes de montaña
  [0.20]  Camelia Oil                → aceites de camelia
  [0.20]  Sea Breeze                 → reconectar a las personas con la naturaleza a través de productos elaborados con ingredientes atlánticos
  [0.15]  Earth Soil                 → la conexión con lo salvaje y lo esencial


## 3. Paso 2 — Embedding por atributo: top-K descriptores Dravnieks

In [11]:
def buscar_descriptores(atributos: list[dict], top_k: int = TOP_K_DESCRIPTORES) -> list[dict]:
    """
    Para cada atributo, busca los top_k descriptores Dravnieks más similares.
    Añade al dict la lista de hits: [{'descriptor': str, 'coseno': float}]
    """
    textos = [a['atributo'] for a in atributos]
    embeddings = embed_model.encode(textos, normalize_embeddings=True, show_progress_bar=False)

    resultado = []
    for i, atrib in enumerate(atributos):
        hits = qdrant.query_points(
            collection_name=COLLECTION,
            query=embeddings[i].tolist(),
            limit=top_k
        ).points
        resultado.append({
            **atrib,
            "descriptores": [
                {"descriptor": h.payload["descriptor"], "coseno": round(h.score, 4)}
                for h in hits
            ]
        })
    return resultado


atributos_con_descriptores = buscar_descriptores(atributos, top_k=TOP_K_DESCRIPTORES)

print("Descriptores Dravnieks por atributo:\n")
for a in atributos_con_descriptores:
    print(f"► [{a['peso']:.2f}] {a['atributo']}")
    for d in a['descriptores']:
        print(f"       {d['descriptor']:30s}  coseno={d['coseno']}")
    print()

Descriptores Dravnieks por atributo:

► [0.25] Marine Algae
       FISHY                           coseno=0.5427
       COLOGNE                         coseno=0.4607
       AROMATIC                        coseno=0.4484
       KIPPERY,SMOKED FISH             coseno=0.4484

► [0.20] Mountain Moss
       MUSTY, EARTHY, MOLDY            coseno=0.5312
       PEAR                            coseno=0.501
       STRAWBERRY                      coseno=0.4956
       GERANIUM LEAVES                 coseno=0.4942

► [0.20] Camelia Oil
       TURPENTINE, PINE OIL            coseno=0.6133
       CARAMEL                         coseno=0.4978
       COCONUT                         coseno=0.4812
       URINE                           coseno=0.4506

► [0.20] Sea Breeze
       WARM                            coseno=0.4583
       LEATHER                         coseno=0.4543
       WET PAPER                       coseno=0.4469
       FISHY                           coseno=0.4207

► [0.15] Earth Soil
     

## 4. Paso 3 — Score ponderado por molécula

Para cada molécula y cada par (atributo, descriptor):

```
contribución = peso_atributo × coseno(atributo, descriptor) × score_dravnieks(molécula, descriptor) / 100
```

El score final de la molécula es la suma de todas las contribuciones.

In [12]:
def calcular_scores_moleculas(atributos_con_descriptores: list[dict],
                               behavior_named: pd.DataFrame,
                               descriptors: list[str]) -> pd.DataFrame:
    """
    Calcula el score final para cada molécula y guarda el desglose completo.
    """
    n_mols = len(behavior_named)
    mol_names = behavior_named['molecule_name'].values

    scores_totales = np.zeros(n_mols)
    desglose = defaultdict(list)

    for atrib in atributos_con_descriptores:
        peso_atrib = atrib['peso']

        for d in atrib['descriptores']:
            desc_name = d['descriptor']
            coseno    = d['coseno']

            if desc_name not in behavior_named.columns:
                continue

            scores_desc = behavior_named[desc_name].values  
            contribucion = peso_atrib * coseno * scores_desc / 100
            scores_totales += contribucion

            for mol_idx, contrib in enumerate(contribucion):
                if contrib > 0.001:  
                    desglose[mol_idx].append({
                        'atributo':       atrib['atributo'],
                        'peso_atributo':  peso_atrib,
                        'descriptor':     desc_name,
                        'coseno':         round(coseno, 4),
                        'score_dravnieks': round(float(scores_desc[mol_idx]), 1),
                        'contribucion':   round(float(contrib), 5)
                    })

    rows = []
    for idx in range(n_mols):
        rows.append({
            'mol_idx':       idx,
            'molecule_name': mol_names[idx],
            'score_total':   round(float(scores_totales[idx]), 5),
            'desglose':      sorted(desglose[idx], key=lambda x: -x['contribucion'])
        })

    df = pd.DataFrame(rows).sort_values('score_total', ascending=False).reset_index(drop=True)
    return df


resultados = calcular_scores_moleculas(atributos_con_descriptores, behavior_named, descriptors)
print(f"Scores calculados para {len(resultados)} moléculas.")
print(f"\nTop 5 preview:")
print(resultados[['molecule_name', 'score_total']].head(5).to_string(index=False))

Scores calculados para 160 moléculas.

Top 5 preview:
     molecule_name  score_total
    TrimethylAmine      0.19383
 HexylAminelowconc      0.18517
HexylAminehighconc      0.16366
        Eucalyptol      0.14183
       MuskTonalid      0.12676


## 5. Paso 4 — Lista de las recomendaciones con desglose de trazabilidad

In [13]:
def mostrar_leaderboard(resultados: pd.DataFrame, top_n: int = TOP_N_MOLECULAS):
    """Imprime la lista con desglose por molécula."""
    top = resultados.head(top_n)
    score_max = top.iloc[0]['score_total']

    print("=" * 65)
    print("  LISTA DE AROMAS — IDENTIDAD OLFATIVA")
    print("=" * 65)

    for rank, row in top.iterrows():
        nombre = row['molecule_name']
        score  = row['score_total']
        barra  = "█" * int(score / score_max * 30)
        pct    = score / score_max * 100

        print(f"\n#{rank+1:02d}  {nombre}")
        print(f"     Score: {score:.5f}  [{barra:<30s}] {pct:.0f}%")

        top_contribs = row['desglose'][:3]
        if top_contribs:
            print("     Activado por:")
            for c in top_contribs:
                print(f"       · atributo '{c['atributo']}' (peso {c['peso_atributo']:.2f}) "
                      f"→ descriptor '{c['descriptor']}' "
                      f"[coseno={c['coseno']}, Dravnieks={c['score_dravnieks']}%] "
                      f"contrib={c['contribucion']:.5f}")

    print("\n" + "=" * 65)


mostrar_leaderboard(resultados, top_n=TOP_N_MOLECULAS)

  LISTA DE AROMAS — IDENTIDAD OLFATIVA

#01  TrimethylAmine
     Score: 0.19383  [██████████████████████████████] 100%
     Activado por:
       · atributo 'Marine Algae' (peso 0.25) → descriptor 'FISHY' [coseno=0.5427, Dravnieks=62.5%] contrib=0.08478
       · atributo 'Sea Breeze' (peso 0.20) → descriptor 'FISHY' [coseno=0.4207, Dravnieks=62.5%] contrib=0.05258
       · atributo 'Marine Algae' (peso 0.25) → descriptor 'KIPPERY,SMOKED FISH' [coseno=0.4484, Dravnieks=18.2%] contrib=0.02039

#02  HexylAminelowconc
     Score: 0.18517  [████████████████████████████  ] 96%
     Activado por:
       · atributo 'Marine Algae' (peso 0.25) → descriptor 'FISHY' [coseno=0.5427, Dravnieks=46.7%] contrib=0.06331
       · atributo 'Sea Breeze' (peso 0.20) → descriptor 'FISHY' [coseno=0.4207, Dravnieks=46.7%] contrib=0.03926
       · atributo 'Marine Algae' (peso 0.25) → descriptor 'KIPPERY,SMOKED FISH' [coseno=0.4484, Dravnieks=21.3%] contrib=0.02391

#03  HexylAminehighconc
     Score: 0.16366  [

## 6. Paso 5 — Explicación en lenguaje natural (Groq)

In [14]:
def generar_explicacion(texto_empresa: str,
                         atributos: list[dict],
                         resultados: pd.DataFrame,
                         top_n: int = 5) -> str:
    """
    Pide al LLM que explique en lenguaje natural por qué estas moléculas
    representan la identidad olfativa de la empresa.
    """
    client = Groq(api_key=GROQ_API_KEY)

    top_mols = resultados.head(top_n)

    # Preparar resumen de resultados para el prompt
    resumen_mols = []
    for _, row in top_mols.iterrows():
        top_descs = [c['descriptor'] for c in row['desglose'][:3]]
        resumen_mols.append(
            f"- {row['molecule_name']} (score={row['score_total']:.4f}, "
            f"descriptores clave: {', '.join(top_descs)})"
        )

    resumen_atribs = [
        f"- '{a['atributo']}' (peso={a['peso']:.2f}): {a['razon']}"
        for a in atributos
    ]

    prompt = f"""Eres un experto en marketing olfativo y branding sensorial.

Empresa analizada:
{texto_empresa.strip()}

Atributos sensoriales identificados:
{chr(10).join(resumen_atribs)}

Moléculas olfativas recomendadas (ranking):
{chr(10).join(resumen_mols)}

Escribe un análisis claro y académico (máx. 250 palabras) explicando:
1. Por qué estas moléculas representan la identidad de la empresa
2. Cómo cada molécula conecta con uno o varios atributos identificados
3. Qué experiencia olfativa conjunta transmiten al consumidor

Tono: rigoroso pero accesible. En español."""

    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.4,
        max_tokens=500
    )

    return response.choices[0].message.content.strip()


print("Generando explicación con Groq...\n")
explicacion = generar_explicacion(texto_empresa, atributos, resultados, top_n=5)
print(explicacion)

Generando explicación con Groq...

**Análisis de la Identidad Olfativa de la Empresa**

La selección de moléculas olfativas recomendadas refleja la esencia de la marca de cosmética natural fundada en Galicia. Estas moléculas representan la identidad de la empresa debido a su conexión con la naturaleza y los ingredientes atlánticos puros.

1. **Representación de la identidad**: Las moléculas como TrimethylAmine, HexylAminelowconc y HexylAminehighconc, con descriptores clave como "FISHY" y "SMOKED FISH", evocan la frescura y la pureza del mar, conectando con el atributo "Marine Algae" (peso=0.25). Por otro lado, Eucalyptol y MuskTonalid, con descriptores como "TURPENTINE" y "AROMATIC", se relacionan con la naturaleza y la esencia de la tierra, conectando con los atributos "Mountain Moss" (peso=0.20) y "Earth Soil" (peso=0.15).
2. **Conexión con atributos**: Cada molécula se conecta con uno o varios atributos identificados. Por ejemplo, TrimethylAmine y HexylAminehighconc se relacionan co

## 7. Pipeline todo-en-uno

In [15]:
def analizar_empresa(texto: str,
                     n_atributos:    int = N_ATRIBUTOS,
                     top_k_desc:     int = TOP_K_DESCRIPTORES,
                     top_n_mols:     int = TOP_N_MOLECULAS) -> dict:
    """
    Pipeline completo. Devuelve dict con:
      - atributos: lista con pesos y razones
      - atributos_con_descriptores: atributos + descriptores Dravnieks matcheados
      - resultados: DataFrame con leaderboard completo
      - explicacion: texto en lenguaje natural
    """
    print("[1/4] Extrayendo atributos con Groq...")
    atributos = extraer_atributos_con_pesos(texto, n=n_atributos)

    print("[2/4] Buscando descriptores Dravnieks por similitud coseno...")
    atribs_desc = buscar_descriptores(atributos, top_k=top_k_desc)

    print("[3/4] Calculando scores ponderados por molécula...")
    resultados = calcular_scores_moleculas(atribs_desc, behavior_named, descriptors)

    print("[4/4] Generando explicación...")
    explicacion = generar_explicacion(texto, atributos, resultados, top_n=5)

    print("\nAnálisis completo.\n")
    mostrar_leaderboard(resultados, top_n=top_n_mols)
    print("\n── EXPLICACIÓN ──────────────────────────────────────────")
    print(explicacion)

    return {
        "atributos":                 atributos,
        "atributos_con_descriptores": atribs_desc,
        "resultados":                resultados,
        "explicacion":               explicacion
    }


# ── Ejecutar el pipeline ─────────────────────────────────────────────────────
output = analizar_empresa(texto_empresa)

[1/4] Extrayendo atributos con Groq...
[2/4] Buscando descriptores Dravnieks por similitud coseno...
[3/4] Calculando scores ponderados por molécula...
[4/4] Generando explicación...

Análisis completo.

  LISTA DE AROMAS — IDENTIDAD OLFATIVA

#01  TrimethylAmine
     Score: 0.18011  [██████████████████████████████] 100%
     Activado por:
       · atributo 'Marine Algae' (peso 0.25) → descriptor 'FISHY' [coseno=0.5427, Dravnieks=62.5%] contrib=0.08478
       · atributo 'Sea Breeze' (peso 0.15) → descriptor 'FISHY' [coseno=0.4207, Dravnieks=62.5%] contrib=0.03943
       · atributo 'Marine Algae' (peso 0.25) → descriptor 'KIPPERY,SMOKED FISH' [coseno=0.4484, Dravnieks=18.2%] contrib=0.02039

#02  HexylAminelowconc
     Score: 0.17650  [█████████████████████████████ ] 98%
     Activado por:
       · atributo 'Marine Algae' (peso 0.25) → descriptor 'FISHY' [coseno=0.5427, Dravnieks=46.7%] contrib=0.06331
       · atributo 'Sea Breeze' (peso 0.15) → descriptor 'FISHY' [coseno=0.4207, Dravn

In [16]:
output['resultados'][['molecule_name', 'score_total']].head(TOP_N_MOLECULAS).to_csv(
    "resultado_leaderboard.csv", index=False
)

# Exportar atributos
pd.DataFrame(output['atributos']).to_csv("resultado_atributos.csv", index=False)

# Exportar explicación
with open("resultado_explicacion.txt", "w", encoding="utf-8") as f:
    f.write(output['explicacion'])

print("Archivos exportados: resultado_leaderboard.csv, resultado_atributos.csv, resultado_explicacion.txt")

Archivos exportados: resultado_leaderboard.csv, resultado_atributos.csv, resultado_explicacion.txt
